# 5.19 — Scaling Numerical Features Using `StandardScaler`

Many models (logistic regression, SVMs, kNN, neural nets) are sensitive to feature scale.

This project scales **numerical features only** using scikit-learn’s `StandardScaler`, and encodes categoricals using `OneHotEncoder`, combined via a `ColumnTransformer`.

Key rule (leakage prevention):

- **Split first**
- **Fit the scaler on `X_train` only**
- **Transform `X_test` using the already-fitted scaler**

In [ ]:
import numpy as np
import pandas as pd

from src.data_loader import load_csv
from src.config import (
    ALL_FEATURES,
    CATEGORICAL_FEATURES,
    EXCLUDED_COLUMNS,
    NUMERICAL_FEATURES,
    TARGET_COLUMN,
    TARGET_SOURCE_COLUMN,
)
from src.preprocessing import (
    separate_features_and_target,
    split_data,
    fit_preprocessor,
    transform_features,
)

## 1) Load data and derive the demo target

The demo dataset uses `yield_kg` to derive a binary classification label (`target`).

`yield_kg` is a **leakage source** and is excluded from features.

In [ ]:
df = load_csv("data/raw/source_demo_crops_20260321.csv")

working = df.copy()
working[TARGET_COLUMN] = (working[TARGET_SOURCE_COLUMN] >= working[TARGET_SOURCE_COLUMN].median()).astype(int)

print("shape:", working.shape)
print("target distribution (normalized):")
print(working[TARGET_COLUMN].value_counts(normalize=True))

## 2) Separate `X` and `y` using explicit feature lists

Feature type lists are the project source of truth:

- `NUMERICAL_FEATURES` → scaled
- `CATEGORICAL_FEATURES` → one-hot encoded
- `EXCLUDED_COLUMNS` → never used as inputs

In [ ]:
X, y = separate_features_and_target(
    working,
    target_column=TARGET_COLUMN,
    feature_columns=[c for c in ALL_FEATURES if c in working.columns],
    excluded_columns=EXCLUDED_COLUMNS,
    verbose=True,
)

print("Numerical:", NUMERICAL_FEATURES)
print("Categorical:", CATEGORICAL_FEATURES)
X.head()

## 3) Correct workflow: split first, then fit `StandardScaler` on training only

Internally, `fit_preprocessor(...)` builds a `ColumnTransformer` like:

- `("num", StandardScaler(), NUMERICAL_FEATURES)`
- `("cat", OneHotEncoder(...), CATEGORICAL_FEATURES)`

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

bundle = fit_preprocessor(
    X_train,
    numeric_features=NUMERICAL_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
)

X_train_t = transform_features(X_train, bundle)
X_test_t = transform_features(X_test, bundle)

print("Transformed shapes:")
print("  X_train_t:", X_train_t.shape)
print("  X_test_t:", X_test_t.shape)
print("Transformed columns:")
print(list(X_train_t.columns))

## 4) Verify scaling behavior (training split only)

For the **scaled numerical columns**, the transformed training data should have:

- mean $ 0$
- standard deviation $ 1$

(Allow small floating-point deviations.)

In [ ]:
# After transformation, scaled numeric feature names remain the original numeric column names.
scaled_numeric_columns = [c for c in NUMERICAL_FEATURES if c in X_train_t.columns]

train_means = X_train_t[scaled_numeric_columns].mean(axis=0)
train_stds = X_train_t[scaled_numeric_columns].std(axis=0, ddof=0)  # StandardScaler uses population std

print("Scaled numeric columns:", scaled_numeric_columns)
print("Train means (should be ~0):")
print(train_means)
print("Train stds (should be ~1):")
print(train_stds)

## 5) What NOT to do (leakage): fit scaling on full data before splitting

This is intentionally incorrect:

- Fitting on full `X` lets test distribution influence the scaler.

You may see the transformed **training** subset no longer has mean 0 / std 1, because the scaler was fit using information outside the training set.

In [ ]:
# WRONG: fit preprocessor on the full dataset (leakage)
leaky_bundle = fit_preprocessor(
    X,
    numeric_features=NUMERICAL_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
)

X_all_t = transform_features(X, leaky_bundle)
X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = split_data(
    X_all_t, y, test_size=0.2, random_state=42
)

scaled_numeric_columns_leaky = [c for c in NUMERICAL_FEATURES if c in X_train_leaky.columns]

print("Leaky train means (often not ~0):")
print(X_train_leaky[scaled_numeric_columns_leaky].mean(axis=0))
print("Leaky train stds (often not ~1):")
print(X_train_leaky[scaled_numeric_columns_leaky].std(axis=0, ddof=0))

## Takeaways

- Scale **only numerical features**.
- Always **split before scaling**.
- Only call `.fit(...)` / `.fit_transform(...)` on **training data**.
- In this repo, the fitted scaler is stored inside the persisted preprocessing bundle (`models/preprocessor.pkl`).